In [ ]:
%pip -q install ultralytics opencv-python-headless shapely pyyaml pandas matplotlib tqdm

from pathlib import Path
import torch

print('Inputs:', [p.name for p in Path('/kaggle/input').iterdir()])
assert torch.cuda.is_available(), 'Enable GPU: Settings → Accelerator → GPU, then Restart'
print('GPU:', torch.cuda.get_device_name(0))

# >>> Edit these two paths to the slugs shown under /kaggle/input <<<
CODE_INPUT = Path('/kaggle/input/parking-detection-code')
DATA_INPUT = Path('/kaggle/input/parking-dataset-2')

In [ ]:
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

EPOCHS = 50
BATCH = 16
MODEL = 'yolov8s.pt'
IMGSZ = 640
PATIENCE = 15
SEED = 42
WORKERS = 2

def extract_single_archive(path: Path, destination: Path) -> Path:
    if path.is_file() and zipfile.is_zipfile(path):
        destination.mkdir(parents=True, exist_ok=True)
        shutil.unpack_archive(str(path), str(destination))
        return destination
    archives = list(path.glob('*.zip')) if path.is_dir() else []
    if len(archives) == 1:
        destination.mkdir(parents=True, exist_ok=True)
        shutil.unpack_archive(str(archives[0]), str(destination))
        return destination
    return path

def find_project_root(path: Path) -> Path:
    candidates = [path, *path.rglob('train.py')]
    for c in candidates:
        if c.name == 'train.py' and c.parent.name == 'src':
            return c.parent.parent
        if (c / 'src' / 'train.py').is_file():
            return c
    raise FileNotFoundError(f'Could not find src/train.py under {path}')

def find_dataset_root(path: Path) -> Path:
    for candidate in [path, *path.rglob('data.yaml')]:
        root = candidate.parent if candidate.name == 'data.yaml' else candidate
        if all((root / split / 'images').is_dir() and (root / split / 'labels').is_dir()
               for split in ('train', 'valid', 'test')):
            return root
    raise FileNotFoundError(f'Expected train/valid/test images+labels under {path}')

CODE_INPUT = extract_single_archive(CODE_INPUT, Path('/kaggle/working/code-input'))
DATA_INPUT = extract_single_archive(DATA_INPUT, Path('/kaggle/working/data-input'))
CODE_ROOT = find_project_root(CODE_INPUT)
RAW_DATA = find_dataset_root(DATA_INPUT)
PROJECT = Path('/kaggle/working/parking-detection')
PROJECT.mkdir(parents=True, exist_ok=True)

for folder in ('src', 'configs', 'tests'):
    src = CODE_ROOT / folder
    if src.is_dir():
        shutil.copytree(src, PROJECT / folder, dirs_exist_ok=True)

print('Code:', CODE_ROOT)
print('Dataset:', RAW_DATA)
print('Working:', PROJECT)
print(f'Train plan: model={MODEL} epochs={EPOCHS} batch={BATCH} imgsz={IMGSZ}')

def run(cmd):
    print('>>', ' '.join(map(str, cmd)))
    subprocess.run(cmd, cwd=PROJECT, check=True)

In [ ]:
# Validate dataset paths and class mapping; train directly from the attached full dataset.
from ultralytics.data.utils import check_det_dataset

DATA_YAML = RAW_DATA / 'data.yaml'
assert DATA_YAML.is_file(), f'Missing dataset config: {DATA_YAML}'
dataset_info = check_det_dataset(str(DATA_YAML), autodownload=False)
assert dataset_info['nc'] == 2, dataset_info['names']
assert list(dataset_info['names'].values()) == ['space-empty', 'space-occupied'], dataset_info['names']
for split in ('train', 'val', 'test'):
    split_path = Path(dataset_info[split])
    assert split_path.is_dir(), f'Missing {split} images: {split_path}'
    print(split, split_path, 'images=', len(list(split_path.glob('*.jpg'))))
print('Dataset YAML:', DATA_YAML)
print('Classes:', dataset_info['names'])

In [ ]:
# Train SLOT detector — 50 epochs
run([
    sys.executable, 'src/train.py',
    '--data', str(DATA_YAML),
    '--model', MODEL,
    '--name', 'slot_det',
    '--epochs', str(EPOCHS),
    '--batch', str(BATCH),
    '--imgsz', str(IMGSZ),
    '--device', '0',
    '--workers', str(WORKERS),
    '--patience', str(PATIENCE),
    '--seed', str(SEED),
    '--project', str(PROJECT / 'runs'),
])
slot_best = PROJECT / 'weights' / 'slot_det_best.pt'
assert slot_best.is_file(), slot_best
print('SLOT weights:', slot_best, 'size_MB', round(slot_best.stat().st_size / 1e6, 2))

In [ ]:
# Evaluate on held-out TEST split (detection metrics only)
run([
    sys.executable, 'src/evaluation.py',
    '--weights', str(PROJECT / 'weights' / 'slot_det_best.pt'),
    '--data', str(DATA_YAML),
    '--split', 'test',
    '--device', '0',
    '--out', str(PROJECT / 'outputs' / 'evaluation' / 'slot_metrics.json'),
    '--max-images', '100',
])
print('Done. Download from /kaggle/working/parking-detection/:')
print('  weights/slot_det_best.pt')
print('  runs/slot_det/results.csv')
print('  outputs/evaluation/slot_metrics.json')

## After Kaggle — local inference

Copy the downloaded `.pt` files to:

`C:\Users\Yug\Desktop\parking-detection\weights\`

```powershell
cd "C:\Users\Yug\Desktop\parking-detection"
python src\inference.py `
  --source "Parking_dataset_2\test\images" `
  --slot-weights weights\slot_det_best.pt `
  --out outputs\inference\kaggle_50ep
```

The project's default vehicle detector remains the pretrained COCO model. This slot model alone does not learn parking violations. If CUDA OOM occurs, set `BATCH = 8` (or `4`) in the config cell and rerun the slot-training cell.